# Riemannian graph & graph spectrum — from dense to sparse

*Converted from the exploratory script `Main2.py`.*

*Archived source: `Main2.py` @ commit `c36a461` — restore with `git show c36a461:Main2.py`.*


## Goal

Overcome the \(O(n^2)\) cost of the full-matrix prototype by building a
**sparse similarity graph** from the point cloud and computing the graph's
**Laplacian spectrum** (eigenvalues / eigenvectors) efficiently.

- **For biologists:** instead of linking every point to every other point, we
  link each point only to its **nearest neighbours**. This is a much better
  model of a plant: a stem point is connected to the points just around it,
  and the chain of connections traces the plant's architecture.
- **For computer scientists:** a *k-nearest-neighbour graph* is sparse
  (linear in the number of points), so its Laplacian and eigenvectors can be
  computed with fast **sparse** linear algebra (Lanczos / `eigsh`), scaling to
  realistic cloud sizes.

## Implementation overview

1. Load a point cloud.
2. Build the **Riemannian graph** (each point linked to its \(k\) nearest
   neighbours) with the library helper `similarity_graph.create_riemannian_graph`.
3. Compute the **graph spectrum** with `graph_spectrum`.
4. **Export** the eigenvectors back onto the point cloud for visualisation.

> The original script loaded `Data/as_cylinders.ply` and called
> `SGk.graph_spectrum(...)`. During the library refactoring, `graph_spectrum`
> moved from `similarity_graph` to `point_cloud_graph`; we import it from its
> current location (an API update, not a rewrite of the approach). The original
> also wrote the eigenvectors to a file for CloudCompare — here we visualise
> them directly in the notebook.


In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt

# Early library module used by the original script.
import spectral_clustering.similarity_graph as SGk
# graph_spectrum moved modules during the refactoring:
from spectral_clustering.point_cloud_graph import graph_spectrum

### 1. Load the point cloud

A *Chenopodium album* scan (the same shipped cloud as the previous notebook,
but used at full resolution here because the graph method is sparse).

In [ ]:
pcd = o3d.io.read_point_cloud("../data/chenos/cheno_3.ply")
print(pcd)

In [ ]:
points = np.asarray(pcd.points)
print("n points:", len(points))

### 2. Build the Riemannian (k-NN) graph

`create_riemannian_graph` links each point to its \(k\) nearest neighbours.
For \(n\) points with \(k\) neighbours this yields a graph with \(n\) nodes and
about \(n\cdot k\) edges — **sparse**, unlike the full matrix of notebook 1.

In [ ]:
# Each point linked to its 8 nearest neighbours.
G = SGk.create_riemannian_graph(pcd, method="knn", nearest_neighbors=8)
print("nodes:", G.number_of_nodes(), " edges:", G.number_of_edges())

### 3. Graph spectrum

`graph_spectrum` builds the (sparse) Laplacian of \(G\) and computes its
\(k\) smallest (or largest) eigenvalues and eigenvectors. These eigenvectors
live on the points of the cloud and encode its global shape.

In [ ]:
k = 10
keigenval, keigenvec = graph_spectrum(G, k=k, smallest_first=False)
print("eigenvalues shape:", keigenval.shape)
print("eigenvectors shape:", keigenvec.shape)
print("first eigenvalues:", np.sort(keigenval)[:5])

### 4. Visualise the eigenvectors on the cloud

Each eigenvector assigns a scalar value to every point. Colouring the cloud by
these values reveals the plant's structure: for a plant, the *second* smallest
eigenvector (the **Fiedler vector**) typically varies smoothly along the
stem/branches — the basis for branch detection in the next notebook.

In [ ]:
for vec_idx in (0, 1, 2):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    sc = ax.scatter(points[:, 0], points[:, 1], points[:, 2], c=keigenvec[:, vec_idx], cmap="Spectral", s=3)
    fig.colorbar(sc, ax=ax)
    ax.set_title(f"eigenvector {vec_idx}")
    plt.show()

## Takeaway

Working with a **sparse Riemannian graph** makes the spectral computation
feasible on full clouds. The graph's eigenvectors are smooth functions over
the plant and are rich geometric descriptors. The next notebook exploits the
**gradient of the Fiedler vector** to locate the junctions between stem and
branches.